# NLP Practical 4 — Syntactic Analysis

Run in **Google Colab**. Cells with `pip install` only need to run once per session.

**Covers:** CFG + chart parsing, PP-attachment ambiguity, PCFG-based parse ranking, industrial dependency parsing with spaCy, WordNet lookup, and an RTN implemented as a recursive-descent parser.


In [ ]:
!pip install nltk spacy -q
!python -m spacy download en_core_web_sm -q
import nltk
nltk.download("wordnet")


In [ ]:
# ============================================================
# PART A: CFG + chart parsing, and PP-attachment ambiguity
# ============================================================
import nltk
from nltk import CFG

grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP | NP PP
VP -> V NP | V NP PP
PP -> P NP
Det -> 'the' | 'a'
N -> 'man' | 'telescope' | 'park' | 'dog'
V -> 'saw'
P -> 'with' | 'in'
""")

parser = nltk.ChartParser(grammar)
sentence = "the man saw the dog with the telescope".split()

print("PP-attachment ambiguity: 'saw the dog WITH the telescope' has 2 readings --")
print("(a) the man used the telescope to see, or (b) the dog carried the telescope.\n")

trees = list(parser.parse(sentence))
print(f"Number of parses found: {len(trees)}")
for i, t in enumerate(trees, 1):
    print(f"\nParse {i}:")
    print(t)


In [ ]:
# ============================================================
# PART B: PCFG -- rank ambiguous parses by probability
# ============================================================
from nltk import PCFG

pcfg = PCFG.fromstring("""
S -> NP VP [1.0]
VP -> V NP PP [0.6]
VP -> VP PP [0.4]
NP -> Det N [0.7]
NP -> NP PP [0.3]
PP -> P NP [1.0]
Det -> 'the' [1.0]
N -> 'man' [0.4] | 'telescope' [0.3] | 'dog' [0.3]
V -> 'saw' [1.0]
P -> 'with' [1.0]
""")

viterbi_parser = nltk.ViterbiParser(pcfg)
sentence = "the man saw the dog with the telescope".split()
tree = viterbi_parser.parse(sentence)
for t in tree:
    print("Highest-probability parse:")
    print(t)


In [ ]:
# ============================================================
# PART C: Industrial dependency parsing with spaCy
# ============================================================
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp("The man saw the dog with the telescope.")
print(f"{'TOKEN':10s}{'DEP':12s}{'HEAD'}")
for tok in doc:
    print(f"{tok.text:10s}{tok.dep_:12s}{tok.head.text}")

print("\nIndustry link: spaCy's transition-based dependency parser is the direct")
print("production successor to the constituency/PCFG parsers built above.")


In [ ]:
# ============================================================
# PART D: WordNet lookup + RTN-as-recursive-descent-parser (reuse from Practical 1 style)
# ============================================================
from nltk.corpus import wordnet as wn

for syn in wn.synsets("telescope")[:3]:
    print(syn.name(), "-", syn.definition())

# Recursive-descent parser recognizing the same grammar as an RTN would
DET, NOUN, VERB, PREP = {"the", "a"}, {"man", "dog", "telescope", "park"}, {"saw"}, {"with", "in"}

def parse_NP(tokens, i):
    if i >= len(tokens) or tokens[i] not in DET:
        return None
    i += 1
    if i >= len(tokens) or tokens[i] not in NOUN:
        return None
    node = {"NP": [tokens[i-1], tokens[i]]}
    i += 1
    while i < len(tokens) and tokens[i] in PREP:
        prep = tokens[i]; i += 1
        sub = parse_NP(tokens, i)
        if not sub: break
        sub_tree, i = sub
        node.setdefault("PP", []).append({"prep": prep, "NP": sub_tree})
    return node, i

result = parse_NP("the man with the telescope".split(), 0)
print("\nRTN-style recursive-descent parse:", result[0] if result else "REJECTED")
